# Methodology

This proposed workflow consists of 5 stages:

### 1. Preparation

To establish a data management method which never overrwrites data but rather creates a record of all instances of data regarding any single street sign being collected, this research design distinguishes two main entities: **signs** and **observations**. These also represent two sheets within an Excel workbook, one to keep record of every sign in the database and their general location, and one to track each encounter with each sign, along with their location, text content, condition, etc which will be the main working sheet for data processing. The purpose of this is to treat signs and encounters with signs as separate entities so that hypothetically, several encounters with one sign throughout time can be analyzed against each other with relative ease.

#### Example 1: SIGN

| **sign_id**                                           | **sign_township**                   | **sign_notes** |
|-------------------------------------------------------|-------------------------------------|----------------|
| Unique identifier for each sign in the format SG####. | Sign location by township/district. |                |
| SG0001                                                | Xinshi Township                     |                |

#### Example 2: OBSERVATION

| **observation_id**                                          | **sign_id**                    | **sign_photo_filename**                                         | **sign_latitude** | **sign_longitude** | **sign_content**                              | **sign_status**        | **sign_condition**     | **sign_notes** |
|-------------------------------------------------------------|--------------------------------|-----------------------------------------------------------------|-------------------|--------------------|-----------------------------------------------|------------------------|------------------------|----------------|
| Unique identifier for each observation in the format O####. | Sign id of the concerned sign. | Filename of the photo of the sign recorded in this observation. | WGS84 latitude.   | WGS84 longitude.   | Unedited text content of the sign/OCR output. | Present/moved/missing. | Good/satisfactory/bad. |                |
| O0001                                                       | SG0001                         | SG0001.jpg                                                      | 23.10262          | 120.2807           | Ulay 烏來 紀念品                              | Present                | Satisfactory           | Faded, dirty.  |

These are the essential sheets. For additional data storage and analysis, I also made 4 additional sheets: **SURVEY**, **PHOTO**, **360**, and **TEXT**. These are mostly relevant for storing metadata not immediately relevant to data processing, qualitative analysis, and display, at least for now.

#### Example 4: SURVEY

| **survey_id**                                           | **survey_date** | **survey_city** | **survey_district**       | **survey_gpx**    |
|---------------------------------------------------------|-----------------|-----------------|---------------------------|-------------------|
| Unique identifier for each survey (field work session). | Survey date.    | Survey city.    | Survey township/district. | gpx filename      |
| S0001                                                   | 26.08.2026      | Taipei City     | Wenshan District          | S0001_Wenshan.gpx |

#### Example 3: TEXT

| **text_id**                                                                                                             | **sign_id**                    | **sign_observation**                  | **sign_photo_filename**                                | **text_content**                                                          | **iso_language**                 | **iso_script**                 | **translation_eng**                | **translation_zho**                |
|-------------------------------------------------------------------------------------------------------------------------|--------------------------------|---------------------------------------|--------------------------------------------------------|---------------------------------------------------------------------------|----------------------------------|--------------------------------|------------------------------------|------------------------------------|
| Unique identifier for each individual text element, defined as distinct size, font, language, etc, in the format T####. | Sign id of the concerned sign. | Observation id of the concerned sign. | Filename of the photo of the sign displaying the text. | Text content of the sign/OCR output broken down into individual elements. | ISO-639-3 code for the language. | ISO-15924 code for the script. | English translation if applicable. | Chinese translation if applicable. |
| T0001                                                                                                                   | SG0001                         | O0001                                 | SG0001.jpg                                             | 烏來                                                                      | zho                              | hant                           | Wulai                              |                                    |

### 2. Field documentation

A street sign is any material object that indicates or refers to something other than itself; this makes the scope of documentation incredibly large, and the process will inevitably be slow. For this project I used the application Comaps to record my route as I walked and later used Python to match image time stamps to gpx data points which gave me the approximate coordinates of all of my images. This was more efficient than manually recording each one.

The fieldwork process therefore consisted of walking through the site with Comaps running in the background and taking close-up photos of all street signs that I came across. To document the surrounding context of the street signs, I also periodically took 360 captures of the area to support qualitative analysis according to Scollon and Scollon Wong's (2003) framework of meaning in place, i.e signs in their environmental context. 

![Field documentation at Wulai waterfall](IMG_0704.JPG)

<iframe
  src="360/S0002_360_017.html"
  width="100%"
  height="700"
  style="border:none;">
</iframe>

### 3. Data processing



I begin by fetching street sign coordinates by matching my Comaps .gpx-file with the time stamps on my .jpg-files:

In [ ]:
# LOCATING THE IMAGE FILES

from pathlib import Path
import pandas as pd

photo_folder = Path("C:/path/to/photo/folder")

files = list(photo_folder.glob("*"))

records = []

for file in files:
    records.append({
        "filename": file.name,
        "path": str(file)
    })

import re

manifest = pd.DataFrame(records)

manifest.head(10)

In [ ]:
# CREATING CSV OF THE FILENAMES TO COPY INTO MY DATA .XLSX

manifest = pd.DataFrame({
    "path": files
})

manifest["filename"] = manifest["path"].apply(
    lambda x: x.name
)

manifest.head(30)

manifest.to_csv("filenames")

In [ ]:
# FETCHING IMAGE TIMESTAMPS

from PIL import Image
from PIL.ExifTags import IFD

def get_timestamp(file_path):
    try:
        with Image.open(file_path) as img:
            exif = img.getexif()

            exif_ifd = exif.get_ifd(IFD.Exif)

            timestamp = exif_ifd.get(36867)

            return timestamp

    except Exception:
        return None

manifest["capture_date_time_original"] = (manifest["path"].apply(get_timestamp))

manifest

manifest.to_csv(capture_timestamps")

In [ ]:
# CONVERTING .GPX DATA POINTS INTO A PANDAS DATA FRAME AND CONVERTING TO TAIWAN TIME

import gpxpy

gpx_path = Path(
    "path/to/gpx"
)

with open(gpx_path, "r", encoding="utf-8") as gpx_file:
    gpx = gpxpy.parse(gpx_file)

track_points = []

for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            track_points.append({
                "gpx_time": point.time,
                "latitude": point.latitude,
                "longitude": point.longitude,
                "elevation": point.elevation
            })

gpx_df = pd.DataFrame(track_points)

gpx_df["gpx_time_taiwan"] = (
    pd.to_datetime(gpx_df["gpx_time"], utc=True)
    .dt.tz_convert("Asia/Taipei")
)

gpx_df["gpx_time_local"] = (
    gpx_df["gpx_time_taiwan"]
    .dt.tz_localize(None)
)

gpx_df.head(30)

In [ ]:
# COMBINING INTO ONE DATA FRAME
gpx_df[
    ["gpx_time_taiwan", "latitude", "longitude"]
].head(30)

In [ ]:
manifest = manifest.sort_values(
    "capture_datetime_original"
).reset_index(drop=True)

gpx_df = gpx_df.sort_values(
    "gpx_time_local"
).reset_index(drop=True)

matched = pd.merge_asof(
    manifest,
    gpx_df[
        ["gpx_time_local", "latitude", "longitude"]
    ],
    left_on="capture_datetime_original",
    right_on="gpx_time_local",
    direction="nearest"
)

matched.head(30)

matched.to_csv("photo_coordinates")

### 4. Image processing & text extraction

I used [PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR/blob/main/README.md) to extract text from the images and input the results into an .xlsx-file.

In [ ]:
import sys
print(sys.executable)

import sys
!{sys.executable} -m pip install paddleocr

import sys
!{sys.executable} -m pip install paddlepaddle

from paddleocr import PaddleOCR

In [ ]:
%pip uninstall -y paddleocr paddlepaddle paddlex
%pip install paddlepaddle==3.2.0 paddleocr==3.3.3

In [ ]:
import os
import gc
from pathlib import Path
import pandas as pd

os.environ["FLAGS_enable_pir_api"] = "0"

from paddleocr import PaddleOCR

input_dir = Path(
    r"path/to/image/file"
)

output_excel = (
    r"path/to/desired/xlsx/ouput/location"
)

ocr = PaddleOCR(
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

files = [
    f for f in input_dir.iterdir()
    if f.suffix.lower() in extensions
]

rows = []

for i, file in enumerate(files, start=1):

    print(f"{i}/{len(files)}: {file.name}")

    try:
        results = ocr.predict(
            input=str(file),
            text_det_limit_type="max",
            text_det_limit_side_len=1280
        )

        text_lines = []

        for result in results:
            data = result.json

            rec_texts = data["res"].get("rec_texts", [])
            text_lines.extend(rec_texts)

        rows.append({
            "filename": file.name,
            "text": "\n".join(text_lines)
        })

        del results
        gc.collect()

    except Exception as e:

        print(f"FAILED: {file.name}")

        rows.append({
            "filename": file.name,
            "text": "",
            "error": str(e)
        })

df = pd.DataFrame(rows)

df.to_excel(output_excel, index=False)

print(output_excel)

After I had all of my desired data in separate spreadsheets, I copied them all into my main Excel ecosystem. As I was going along, I was manually cross-referencing the data with my images and making qualitative notes about the sign condition. This was incredibly time consuming, and for future editions I could see parts of this being automated in some way. However, I found this step important for the integrity of the data. I also used this opportunity to make sure none of my images contained any sensitive data. In one instance a sign contained both interesting data for analysis and a personal social media account ID, in which case I used a photo editor to blur out the identifiable data.

### 5. Qualitative analysis

The purpose of this workflow is to streamline the process of documenting street signs in a site in order to identify themes, patterns, and topics of interest for analysis. In the following sections I demonstrate some ways of displaying and searching the data and manually analyzing the text contents of identified signs of interest.